In [1]:
%pip install -q ipywidgets matplotlib numpy
%matplotlib inline
# ============================================================================
#  Synchronous Buck - interactive  (clean rebuild)
#  ipywidgets controls (HTML) + inline re-render of one matplotlib figure.
#  Sliders update on release; type exact values in the boxes. Buttons solve
#  forward/inverse. Run via  Run > Run All Cells.  DEBUG=False silences the log.
# ============================================================================
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import ipywidgets as W
from IPython.display import display

DEBUG = True
def _dbg(*a):
    if DEBUG: print("[buck]", *a)

plt.close('all')   # clean slate on re-run

# ---- fixed params ----
Vin, L, fs, C = 12.0, 10e-6, 100e3, 2.2e-6
Ts = 1.0/fs
N_CYCLES = 3
IL_C='#ff7f0e'; HS_C='#d62728'; LS_C='#e8b200'; VOUT_C='#1f77b4'; LOAD_C='#9467bd'; PWM_C='#2ca02c'

# ============================ MODEL (validated, pure) ========================
def operating_point(D, R, ctrl='DEM', Lval=L):
    K = 2.0*Lval/(R*Ts)
    if ctrl=='FCCM': return D*Vin, D, 'CCM (forced)', K
    if K >= 1.0-D:   return D*Vin, D, 'CCM', K
    M = 2.0/(1.0+np.sqrt(1.0+4.0*K/D**2))
    return M*Vin, M, 'DCM', K

def simulate(D, V0, R, ctrl='DEM', Lval=L, n_cycles=N_CYCLES, spc=1200):
    dt=Ts/spc; N=n_cycles*spc; t=np.arange(N)*dt
    iL=np.zeros(N); vC=np.zeros(N); il, vc = 0.0, float(V0)
    for k in range(N):
        on=(t[k]%Ts) < D*Ts
        if on: dil=(Vin-vc)/Lval
        else:
            dil=-vc/Lval
            if ctrl=='DEM' and il<=0.0: il,dil=0.0,0.0
        dvc=(il - vc/R)/C
        iL[k],vC[k]=il,vc
        il=il+dil*dt
        if ctrl=='DEM' and (not on) and il<0.0: il=0.0
        vc=vc+dvc*dt
    return t, iL, vC

def settled_vcap(D, R, ctrl='DEM', Lval=L):
    _t,_iL,_vC=simulate(D, operating_point(D,R,ctrl,Lval)[0], R, ctrl, Lval, n_cycles=60)
    return float(np.mean(_vC[-1200:]))

def required_duty(Vout_t, R, ctrl, Lval):
    """Duty needed to sustain Vout_t for given L,R,mode. Clamped to [0.05,0.95]."""
    M = min(max(Vout_t/Vin, 1e-3), 0.999)
    K = 2.0*Lval/(R*Ts)
    if ctrl=='FCCM' or K >= 1.0-M:
        D = M
    else:
        denom = ((2.0-M)/M)**2 - 1.0
        D = math.sqrt(4.0*K/denom) if denom > 1e-12 else 0.95
    return min(max(D, 0.05), 0.95)

def ceil_1sf(x):
    if x<=0: return 0.0
    mag=10**math.floor(math.log10(x)); return math.ceil(x/mag)*mag

def solve(D, V0, R, ctrl, Lh):
    """One call -> everything the view needs."""
    t,iL,vC = simulate(D,V0,R,ctrl,Lh)
    on = (t % Ts) < D*Ts
    Vt,M,mode,K = operating_point(D,R,ctrl,Lh)
    iHS=np.where(on,iL,np.nan); iLS=np.where(~on,iL,np.nan); iload=vC/R
    pwm=on.astype(float); hs=on.astype(float)
    ls=(~on).astype(float) if ctrl=='FCCM' else ((~on)&(iL>1e-6)).astype(float)
    return dict(D=D,V0=V0,R=R,ctrl=ctrl,Lh=Lh,t=t*1e6,iL=iL,vC=vC,
                iHS=iHS,iLS=iLS,iload=iload,pwm=pwm,hs=hs,ls=ls,
                Vt=Vt,M=M,mode=mode,K=K)

# ================================ VIEW =====================================
def draw_schematic(ax):
    ax.clear(); ax.axis('off')
    ax.set_xlim(0, 12); ax.set_ylim(0.3, 2.9)
    ax.set_aspect('equal')            # so the summer is a true circle, not an oval
    my = 1.55                          # main signal-flow height
    def box(x,y,w,h,label,color,fs=10):
        ax.add_patch(FancyBboxPatch((x,y),w,h,
            boxstyle='round,pad=0.02,rounding_size=0.08',
            lw=1.6,edgecolor=color,facecolor='white',zorder=2))
        ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=fs,color=color,
                fontweight='bold',zorder=3)
        return dict(e=(x+w,y+h/2), w_=(x,y+h/2), c=(x+w/2,y+h/2), top=(x+w/2,y+h), bot=(x+w/2,y))
    def arrow(p,q,color,lw=1.6):
        ax.add_patch(FancyArrowPatch(p,q,arrowstyle='-|>',mutation_scale=11,
                     lw=lw,color=color,zorder=1,shrinkA=0,shrinkB=0))
    pwm=box(0.3, my-0.45, 1.6, 0.9, 'PWM\n(duty D)', PWM_C, fs=9)
    hs =box(2.7, 1.95, 1.0, 0.55, 'HS', HS_C)
    ls =box(2.7, 0.60, 1.0, 0.55, 'LS', LS_C)
    arrow(pwm['e'], hs['w_'], HS_C); arrow(pwm['e'], ls['w_'], LS_C)
    merge=(4.35, my)
    arrow(hs['e'], merge, HS_C); arrow(ls['e'], merge, LS_C)
    Lb=box(4.7, my-0.45, 0.9, 0.9, 'L', IL_C)
    arrow(merge, Lb['w_'], IL_C)
    sx, r = 6.7, 0.28
    ax.add_patch(plt.Circle((sx,my), r, fill=False, lw=1.6, color='black', zorder=2))
    ax.text(sx, my, '+', ha='center', va='center', fontsize=13, zorder=3)
    arrow(Lb['e'], (sx-r, my), IL_C); ax.text(6.05, my+0.32, 'iL', color=IL_C, fontsize=9)
    rc=box(7.6, my-0.55, 1.7, 1.1, 'R load\n+ C', LOAD_C, fs=9)
    arrow((sx+r, my), rc['w_'], 'black'); ax.text(7.05, my+0.30, 'iC', color='black', fontsize=8)
    vout=(10.9, my)
    arrow(rc['e'], vout, VOUT_C)
    ax.plot([vout[0]],[vout[1]],'o',color=VOUT_C,zorder=4)
    ax.text(10.55, my+0.34, 'Vout', color=VOUT_C, fontsize=10, fontweight='bold')
    # iload: solid arrow straight up into the summer from below
    arrow((sx, 0.62), (sx, my-r), LOAD_C)
    ax.text(sx+0.12, 0.78, 'iload', color=LOAD_C, fontsize=9)

def draw_waveforms(ax_pwm, ax_cur, axv, ax_tf, res, hist, il_lim):
    for ax in (ax_pwm, ax_cur, axv, ax_tf): ax.clear()
    axv.yaxis.set_label_position('right'); axv.yaxis.set_ticks_position('right')
    t=res['t']; nk=len(hist)
    # ---- fading ghost trails of previous states ----
    for gi,h in enumerate(hist):
        a=0.07+0.30*(gi+1)/max(nk,1)
        ax_pwm.plot(h['t'],h['pwm']+2.8,drawstyle='steps-post',color=PWM_C,ls='--',lw=0.9,alpha=a,zorder=1)
        ax_pwm.plot(h['t'],h['hs']+1.4,drawstyle='steps-post',color=HS_C,ls='--',lw=0.9,alpha=a,zorder=1)
        ax_pwm.plot(h['t'],h['ls'],drawstyle='steps-post',color=LS_C,ls='--',lw=0.9,alpha=a,zorder=1)
        ax_cur.plot(h['t'],h['vC'],color=VOUT_C,ls='--',lw=1.0,alpha=a,zorder=1)
        axv.plot(h['t'],h['iL'],color=IL_C,ls='--',lw=0.9,alpha=a,zorder=1)
        axv.plot(h['t'],h['iHS'],color=HS_C,ls='--',lw=0.9,alpha=a,zorder=1)
        axv.plot(h['t'],h['iLS'],color=LS_C,ls='--',lw=0.9,alpha=a,zorder=1)
        axv.plot(h['t'],h['iload'],color=LOAD_C,ls=':',lw=0.8,alpha=a,zorder=1)
    # ---- PWM / gates (solid) ----
    ax_pwm.plot(t,res['pwm']+2.8,drawstyle='steps-post',color=PWM_C,lw=1.6)
    ax_pwm.plot(t,res['hs']+1.4,drawstyle='steps-post',color=HS_C,lw=1.6)
    ax_pwm.plot(t,res['ls'],drawstyle='steps-post',color=LS_C,lw=1.6)
    ax_pwm.text(0.4,3.8,'duty',color=PWM_C,fontsize=8,fontweight='bold',va='center')
    ax_pwm.text(0.4,2.4,'ON',color=HS_C,fontsize=8,fontweight='bold',va='center')
    ax_pwm.text(res['D']*Ts*1e6+0.4,1.0,'ON',color=LS_C,fontsize=8,fontweight='bold',va='center')
    ax_pwm.set_ylim(-0.3,4.2); ax_pwm.set_yticks([3.3,1.9,0.5]); ax_pwm.set_yticklabels(['PWM','HS','LS'])
    ax_pwm.set_ylabel('gates'); ax_pwm.grid(alpha=0.3); ax_pwm.tick_params(labelbottom=False)
    ax_pwm.set_title(f"{res['mode']}:  D={res['D']:.2f}   R={res['R']:.3g} \u03a9   \u2192   Vout \u2248 {res['Vt']:.2f} V", fontsize=10)
    # ---- Vout (left) + currents (right), solid ----
    ax_cur.plot(t,res['vC'],color=VOUT_C,lw=2.0,label='Vout')
    ax_cur.axhline(res['Vt'],color=VOUT_C,ls=':',lw=1.2,alpha=0.7)
    ax_cur.set_ylabel('Vout [V]',color=VOUT_C); ax_cur.tick_params(axis='y',labelcolor=VOUT_C)
    ax_cur.set_ylim(0,Vin*1.1); ax_cur.set_xlabel('time [\u00b5s]'); ax_cur.set_xlim(0,N_CYCLES*Ts*1e6)
    ax_cur.grid(alpha=0.3); ax_cur.legend(loc='upper left',fontsize=8)
    axv.plot(t,res['iL'],color=IL_C,dashes=(2,5),lw=2.0,zorder=6,label='iL')
    axv.plot(t,res['iHS'],color=HS_C,lw=2.0,alpha=0.9,label='i_HS')
    axv.plot(t,res['iLS'],color=LS_C,lw=2.0,alpha=0.95,label='i_LS')
    axv.plot(t,res['iload'],color=LOAD_C,ls='--',lw=1.6,zorder=3,label='i_load')
    axv.axhline(0,color='gray',lw=0.7); axv.set_ylabel('current [A]')
    il_lim['hi']=max(il_lim['hi'], ceil_1sf(max(0.0, float(res['iL'].max()))))
    il_lim['lo']=min(il_lim['lo'],-ceil_1sf(max(0.0,-float(res['iL'].min()))))
    axv.set_ylim(il_lim['lo'], il_lim['hi'] if il_lim['hi']>0 else 1.0)
    axv.legend(loc='upper right',fontsize=8,ncol=2)
    # ---- DC transfer M vs D ----
    Dsw=np.linspace(0.02,0.98,300)
    Msw=np.array([operating_point(dd,res['R'],res['ctrl'],res['Lh'])[1] for dd in Dsw])
    ax_tf.plot(Dsw,Msw,color='tab:green',lw=2.0,label='M')
    ax_tf.plot(Dsw,Dsw,'--',color='gray',lw=1.2,label='M=D')
    ax_tf.plot(res['D'],res['M'],'o',color=VOUT_C,ms=8,zorder=5)
    if res['ctrl']=='DEM' and 0<1-res['K']<1: ax_tf.axvline(1-res['K'],color='tab:orange',ls=':',lw=1.2)
    ax_tf.set_xlabel('duty D'); ax_tf.set_ylabel('M = Vout/Vin')
    ax_tf.set_xlim(0,1); ax_tf.set_ylim(0,1); ax_tf.grid(alpha=0.3); ax_tf.legend(loc='upper left',fontsize=8)

# ============================ FIGURE (built once) ==========================
fig = plt.figure(figsize=(8.2, 7.6))
gs  = fig.add_gridspec(3, 2, height_ratios=[0.85, 0.85, 1.25], hspace=0.5, wspace=0.30)
ax_sch = fig.add_subplot(gs[0, :])
ax_pwm = fig.add_subplot(gs[1, 0])
ax_cur = fig.add_subplot(gs[2, 0])
ax_tf  = fig.add_subplot(gs[1:, 1])
axv    = ax_cur.twinx()
plt.close(fig)                 # don't let inline auto-show the blank fig
draw_schematic(ax_sch)         # static: drawn once

# ================================ STATE ====================================
_hist=[]; MAXGHOST=5
_il_lim={'hi':0.0,'lo':0.0}
_suspend={'on':False}

# ================================ CONTROLS =================================
D_DEF, L_DEF, R_DEF = 0.50, 10.0, 20.0
V_DEF = round(settled_vcap(D_DEF, R_DEF, 'DEM', L_DEF*1e-6), 1)

sl_D = W.FloatSlider(value=D_DEF, min=0.05, max=0.95, step=0.01, readout=False, continuous_update=False)
sl_L = W.FloatLogSlider(value=L_DEF, base=10, min=0, max=2, step=0.01, readout=False, continuous_update=False)
sl_R = W.FloatLogSlider(value=R_DEF, base=10, min=0, max=6, step=0.01, readout=False, continuous_update=False)
sl_V = W.FloatSlider(value=V_DEF, min=0, max=12, step=0.1, readout=False, continuous_update=False)
for s in (sl_D, sl_L, sl_R, sl_V): s.layout = W.Layout(width='200px')

tx_D = W.BoundedFloatText(value=D_DEF, min=0.05, max=0.95, step=0.01, layout=W.Layout(width='90px'))
tx_L = W.BoundedFloatText(value=L_DEF, min=1, max=100,  step=0.1, layout=W.Layout(width='90px'))
tx_R = W.BoundedFloatText(value=R_DEF, min=1, max=1e6,  step=1,   layout=W.Layout(width='100px'))
tx_V = W.BoundedFloatText(value=V_DEF, min=0, max=12,   step=0.1, layout=W.Layout(width='90px'))
for s,t in [(sl_D,tx_D),(sl_L,tx_L),(sl_R,tx_R),(sl_V,tx_V)]:
    W.jslink((s,'value'),(t,'value'))   # in-browser sync: drag OR type

mode = W.ToggleButtons(options=[('DEM (DCM)','DEM'),('FCCM (CCM)','FCCM')], value='DEM')
b_vout = W.Button(description='Update Vout', tooltip='Set Vout to the settled output for this D')
b_duty = W.Button(description='Update Duty', tooltip='Set D to the duty that sustains this Vout')
b_reset= W.Button(description='Reset', button_style='warning')
out = W.Output()

def _lab(text, color, width):
    return W.HTML(f"<span style='color:{color};font-weight:600'>{text}</span>",
                  layout=W.Layout(width=width))
def _row(name, sl, tx, unit, color):
    return W.HBox([_lab(name, color, '95px'), sl, tx, _lab(unit, color, '30px')],
                  layout=W.Layout(align_items='center'))
panel = W.VBox([
    _row('Duty D',    sl_D, tx_D, '',        PWM_C),
    _row('L',         sl_L, tx_L, '\u00b5H', IL_C),
    _row('R load',    sl_R, tx_R, '\u03a9',  LOAD_C),
    _row('Vout (V0)', sl_V, tx_V, 'V',        VOUT_C),
    W.HBox([_lab('LS mode', 'black', '95px'), mode]),
    W.HBox([b_vout, b_duty, b_reset]),
])

# ================================ CONTROLLER ===============================
def _read():
    return sl_D.value, sl_V.value, sl_R.value, mode.value, sl_L.value*1e-6

def update(*_):
    if _suspend['on']: return
    res = solve(*_read())
    draw_waveforms(ax_pwm, ax_cur, axv, ax_tf, res, _hist, _il_lim)
    with out:
        out.clear_output(wait=True)
        display(fig)
    _hist.append(res)
    if len(_hist) > MAXGHOST: _hist.pop(0)

def on_vout(_):
    v = settled_vcap(sl_D.value, sl_R.value, mode.value, sl_L.value*1e-6)
    sl_V.value = round(min(max(v,0.0),12.0), 1)          # fires update

def on_duty(_):
    d = required_duty(sl_V.value, sl_R.value, mode.value, sl_L.value*1e-6)
    sl_D.value = round(d, 2)                              # fires update

def on_reset(_):
    _suspend['on'] = True
    sl_D.value, sl_L.value, sl_R.value, sl_V.value, mode.value = D_DEF, L_DEF, R_DEF, V_DEF, 'DEM'
    _il_lim['hi']=0.0; _il_lim['lo']=0.0; _hist.clear()
    _suspend['on'] = False
    update()

for s in (sl_D, sl_L, sl_R, sl_V): s.observe(update, names='value')
mode.observe(update, names='value')
b_vout.on_click(on_vout); b_duty.on_click(on_duty); b_reset.on_click(on_reset)

# ================================ GO =======================================
display(W.HBox([out, panel], layout=W.Layout(align_items='flex-start')))
update()
_dbg(f"matplotlib {plt.matplotlib.__version__} | ipywidgets {W.__version__} | inline render ready")


You should consider upgrading via the '/Users/boren_wang/.venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


[buck] matplotlib 3.9.4 | ipywidgets 8.1.8 | inline render ready


---
### About this tool

A **synchronous buck** (two ideal switches) simulated as a real time-stepping transient, with **ipywidgets** controls and an inline-rendered figure. The LS **control law** sets the mode: *DEM* → DCM possible (LS gate cuts off at the zero-crossing, the idle gap in the PWM strip), *FCCM* → always CCM (iL can reverse).

**Controls** (drag a slider — it updates on release — or type an exact value in the box):

- **Duty D**, **L** (µH, log), **R load** (Ω, log), **Vout (V0)** (V).
- **LS mode**: DEM (DCM) / FCCM (CCM).
- **Update Vout** — set V0 to the settled output for the current D, L, R, mode.
- **Update Duty** — the inverse: set D to the duty that sustains the V0 you chose (clamped to 0.05–0.95).
- **Reset** — back to defaults and clears the ghost trails.

**Figure:** the schematic on top; then the **PWM gates** (green = *duty*, red = HS, amber = LS); the **Vout (left, blue) / currents (right)** panel; and the **DC transfer** M–vs–D with the moving operating point. Both time-domain panels keep **fading dashed ghost trails** of previous states as you sweep (Reset clears them).

**Running:** `Run ▸ Run All Cells`. The first line `%matplotlib inline` is what makes the inline render reliable — no live-canvas widget to drop. Set `DEBUG = False` near the top to silence the status line.
